# nbtools

> Read, grep, and edit Jupyter notebook cells directly (by cell id), without hand-rolling JSON surgery each time. Plain functions -- import them directly, drive them from the `slmn` CLI, or expose them over MCP (see `03_mcp.ipynb`).

In [ ]:
#| default_exp nbtools

In [ ]:
#| export
import json, re, os
from pathlib import Path
import nbformat as _nbf

In [ ]:
#| export
def _resolve_safe(path:str) -> str:
    "Resolve `path`, enforcing SLMN_SANDBOX_ROOTS if set (colon-separated allowed directories, e.g. '/home/user/notebooks:/tmp'). Off by default: with no env var, any path is allowed, same as before sandboxing existed. Raises PermissionError if the resolved path falls outside every allowed root. Every path-taking tool in this module calls this first."
    resolved = Path(path).resolve()
    roots = os.environ.get('SLMN_SANDBOX_ROOTS')
    if not roots: return str(resolved)
    allowed = [Path(r).resolve() for r in roots.split(':') if r]
    if not any(resolved.is_relative_to(root) for root in allowed):
        raise PermissionError(f"{path!r} resolves outside SLMN_SANDBOX_ROOTS ({roots})")
    return str(resolved)

In [ ]:
#| export
def read_nb(path:str, # path to the .ipynb file
            cell_ids:list[str]=None, # if given, only these cells' full source (ids may be partial/substring matches)
            full:bool=False # if True (and no cell_ids given), print full source of every cell
            ) -> str:
    "Read cell source(s) from a Jupyter notebook. With no cell_ids and full=False, lists every cell's id + a source preview (one line each) -- good for getting your bearings before a targeted read."
    path = _resolve_safe(path)
    nb = json.load(open(path))
    cell_ids = cell_ids or []
    out = []
    for cell in nb['cells']:
        cid = cell.get('id', '')
        src = ''.join(cell['source'])
        if full or (cell_ids and any(q in cid for q in cell_ids)):
            out.append(f"=== {cid} ===\n{src}\n")
        elif not cell_ids:
            out.append(f"{cid}  {src[:60].replace(chr(10), ' ')}")
    return '\n'.join(out)

In [ ]:
#| export
def grep_nb(path:str, # path to the .ipynb file
            pattern:str, # regex pattern to search for, per line, within each cell's source
            context:int=0, # lines of context before/after each match to include
            line_numbers:bool=False # prefix each shown line with its (0-based) line number within the cell
            ) -> str:
    "Grep cell sources in a Jupyter notebook. Returns the matching cells (id header + matching/context lines); empty string if nothing matched."
    path = _resolve_safe(path)
    nb = json.load(open(path))
    blocks = []
    for cell in nb['cells']:
        cid = cell.get('id', '')
        src = ''.join(cell.get('source', []))
        lines = src.split('\n')
        match_indices = [i for i, l in enumerate(lines) if re.search(pattern, l)]
        if not match_indices: continue
        shown = set()
        cell_lines = [f"=== {cid} ==="]
        for mi in match_indices:
            lo, hi = max(0, mi - context), min(len(lines) - 1, mi + context)
            if lo > 0 and shown and min(shown) > lo + 1:
                cell_lines.append('--')
            for li in range(lo, hi + 1):
                if li not in shown:
                    prefix = f"{li:4d}: " if line_numbers else ""
                    marker = ">" if li == mi else " "
                    cell_lines.append(f"{marker}{prefix}{lines[li]}")
                    shown.add(li)
        blocks.append('\n'.join(cell_lines))
    return '\n\n'.join(blocks)

In [ ]:
#| export
def edit_nb(path:str, # path to the .ipynb file
            cell_id:str, # cell id, or a prefix of one
            old_str:str, # text to find in the cell's source (first occurrence)
            new_str:str # replacement text
            ) -> str:
    "Replace the first occurrence of old_str with new_str in one cell's source, addressed by id-prefix. Falls back to a trailing-whitespace-insensitive match if the exact text isn't found. Raises ValueError if the cell or the text can't be found."
    path = _resolve_safe(path)
    nb = json.load(open(path))
    cell = next((c for c in nb['cells'] if c.get('id', '').startswith(cell_id)), None)
    if cell is None:
        raise ValueError(f"cell {cell_id!r} not found in {path}")
    src = ''.join(cell['source'])
    if old_str not in src:
        strip_trailing = lambda s: '\n'.join(l.rstrip() for l in s.split('\n'))
        src_stripped, old_stripped = strip_trailing(src), strip_trailing(old_str)
        if old_stripped not in src_stripped:
            raise ValueError(f"old_string not found in cell {cell_id}")
        src, old_str = src_stripped, old_stripped
    cell['source'] = src.replace(old_str, new_str, 1)
    json.dump(nb, open(path, 'w'), indent=1, ensure_ascii=False)
    return f"OK: updated cell {cell_id} in {path}"

In [ ]:
#| export
def patch_nb_cell(path:str, # path to the .ipynb file
                   cell_id:str, # exact cell id
                   new_source:str # the cell's complete new source, replacing the old source entirely
                   ) -> str:
    "Replace a cell's entire source (exact id match, not a prefix -- use this when you want to overwrite a cell wholesale rather than patch part of it; see edit_nb for a targeted string replacement). Raises ValueError if the cell isn't found."
    path = _resolve_safe(path)
    nb = json.load(open(path))
    cell = next((c for c in nb['cells'] if c.get('id') == cell_id), None)
    if cell is None:
        raise ValueError(f"cell {cell_id} not found in {path}")
    cell['source'] = new_source
    json.dump(nb, open(path, 'w'), indent=1, ensure_ascii=False)
    return f"OK: patched cell {cell_id} in {path}"

In [ ]:
#| export
def insert_cells(path:str, # path to the .ipynb file
                  anchor_id:str, # exact cell id to insert after
                  sources:list[str] # one or more new cells' source, inserted in order right after the anchor
                  ) -> str:
    "Insert one or more new code cells immediately after the cell with id `anchor_id`. A thin bulk convenience wrapper over add_nb_cell (the single-cell primitive), so the actual nbformat plumbing lives in one place. Raises ValueError if the anchor isn't found."
    last = anchor_id
    for s in sources:
        last = add_nb_cell(path, s, cell_type='code', after_id=last)
    return f"OK: inserted {len(sources)} cell(s) after {anchor_id} in {path}"

In [ ]:
#| export
def add_nb_cell(path:str, # path to the .ipynb file
                source:str, # the new cell's source text (include any nbdev directives, e.g. a leading '#| export', literally)
                cell_type:str='code', # nbformat cell type: 'code', 'markdown', or 'raw'
                after_id:str=None, # id of the cell to insert immediately after; None appends at the end (in cell order, NOT a raw file append)
                cell_id:str=None # if given, force the new cell's id (e.g. to match an intended nbdev '# %%' marker); default lets nbformat assign one
                ) -> str:
    "Add a new cell to a Jupyter notebook and return its id. Appends to the end by default, or inserts right after `after_id`. `cell_type` is an nbformat type ('code'/'markdown'/'raw') -- note that boopiter-style note/prompt/assistant cells are all 'markdown' at the nbformat level. Reads/writes the file as plain JSON (indent=1) like the other nbtools here, so a one-cell add stays a one-cell git diff rather than reserializing the whole notebook. Raises ValueError on an unknown `cell_type`, or an `after_id` that isn't present."
    path = _resolve_safe(path)
    nb = json.load(open(path))
    makers = {'code': _nbf.v4.new_code_cell, 'markdown': _nbf.v4.new_markdown_cell, 'raw': _nbf.v4.new_raw_cell}
    if cell_type not in makers:
        raise ValueError(f"cell_type must be one of {sorted(makers)}, got {cell_type!r}")
    cell = dict(makers[cell_type](source))  # nbformat builds a schema-valid cell (id/metadata/etc.); keep it as a plain dict
    if cell_id is not None: cell['id'] = cell_id
    cells = nb['cells']
    if after_id is None:
        cells.append(cell)
    else:
        idx = next((i for i, c in enumerate(cells) if c.get('id') == after_id), None)
        if idx is None:
            raise ValueError(f"after_id {after_id!r} not found in {path}")
        cells.insert(idx + 1, cell)
    json.dump(nb, open(path, 'w'), indent=1, ensure_ascii=False)
    return cell['id']


In [ ]:
#| eval: false
# Round-trip smoke test against a throwaway notebook.
import tempfile, os
tmp = tempfile.mktemp(suffix='.ipynb')
_nbf.write(_nbf.v4.new_notebook(cells=[_nbf.v4.new_code_cell('x = 1', id='cell0')]), tmp)

print(insert_cells(tmp, 'cell0', ['y = 2', 'z = 3']))
print(read_nb(tmp))
print(grep_nb(tmp, 'y ='))
print(edit_nb(tmp, 'cell0', 'x = 1', 'x = 100'))
print(patch_nb_cell(tmp, 'cell0', 'x = 999'))
print(read_nb(tmp, full=True))

os.remove(tmp)